# Data Preparation Lab -- Module 2, Class 2

**Dataset:** Superstore Sales

In this lab you will:
1. Load and inspect data
2. Handle missing values
3. Remove duplicates
4. Convert data types
5. Create derived features

The first 3 tasks are pre-built. The rest are TODO for you.

---
## Setup: Load the Dataset

Option A: Upload from Kaggle (download from https://www.kaggle.com/datasets/vivek468/superstore-dataset-final)

Option B: Use the URL loader below.

In [5]:
import pandas as pd
import numpy as np

# Option A: Upload in Colab
# from google.colab import files
# uploaded = files.upload()  # upload your CSV
# df = pd.read_csv('SampleSuperstore.csv')

# Option B: Load from a public URL
# If the URL does not work, use Option A.
url = "https://raw.githubusercontent.com/leonism/sample-superstore/master/data/superstore.csv"

try:
    df = pd.read_csv(url, encoding='latin-1')
    print(f"Loaded from URL: {df.shape[0]} rows, {df.shape[1]} columns")
except Exception as e:
    print(f"URL failed ({e}). Use Option A: upload the CSV manually.")
    # Fallback: upload manually
    from google.colab import files
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    df = pd.read_csv(filename, encoding='latin-1')
    print(f"Loaded from upload: {df.shape[0]} rows, {df.shape[1]} columns")

Loaded from URL: 10800 rows, 21 columns


---
## Task 1: Inspect the Data (pre-built)

Always look at your data before doing anything to it.

In [6]:
# First 5 rows
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2017-152156,11/8/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2.0,0.00,41.9136
1,2,CA-2017-152156,11/8/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3.0,0.00,219.5820
2,3,CA-2017-138688,6/12/2017,6/16/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2.0,0.00,6.8714
3,4,US-2016-108966,10/11/2016,10/18/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5.0,0.45,-383.0310
4,5,US-2016-108966,10/11/2016,10/18/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2.0,0.20,2.5164


In [7]:
# Shape: rows x columns
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")

Shape: 10800 rows, 21 columns


In [8]:
# Data types and non-null counts
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10800 entries, 0 to 10799
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         10800 non-null  str    
 1   Order ID       10800 non-null  str    
 2   Order Date     9994 non-null   str    
 3   Ship Date      9994 non-null   str    
 4   Ship Mode      9994 non-null   str    
 5   Customer ID    9994 non-null   str    
 6   Customer Name  9994 non-null   str    
 7   Segment        9994 non-null   str    
 8   Country        9994 non-null   str    
 9   City           9994 non-null   str    
 10  State          9994 non-null   str    
 11  Postal Code    9983 non-null   float64
 12  Region         9994 non-null   str    
 13  Product ID     9994 non-null   str    
 14  Category       9994 non-null   str    
 15  Sub-Category   9994 non-null   str    
 16  Product Name   9994 non-null   str    
 17  Sales          9994 non-null   float64
 18  Quantity       99

In [9]:
# Summary statistics for numerical columns
df.describe()

,Postal Code,Sales,Quantity,Discount,Profit
count,9983.000000,9994.000000,9994.000000,9994.000000,9994.000000
mean,55245.233297,229.858001,3.789574,0.156203,28.656896
std,32038.715955,623.245101,2.225110,0.206452,234.260108
min,1040.000000,0.444000,1.000000,0.000000,-6599.978000
25%,23223.000000,17.280000,2.000000,0.000000,1.728750
50%,57103.000000,54.490000,3.000000,0.200000,8.666500
75%,90008.000000,209.940000,5.000000,0.200000,29.364000
max,99301.000000,22638.480000,14.000000,0.800000,8399.976000


---
## Task 2: Missing Values (pre-built)

Check which columns have missing values and how many.

In [10]:
# Count missing values per column
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_report = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
})

# Show only columns with missing values
missing_report[missing_report['missing_count'] > 0]

,missing_count,missing_pct
Order Date,806,7.46
Ship Date,806,7.46
Ship Mode,806,7.46
Customer ID,806,7.46
Customer Name,806,7.46
Segment,806,7.46
Country,806,7.46
City,806,7.46
State,806,7.46
Postal Code,817,7.56


In [11]:
# Fill missing numerical values with median (robust to outliers)
numerical_cols = df.select_dtypes(include=[np.number]).columns
for col in numerical_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"Filled {col} missing values with median: {median_val}")

# Fill missing categorical values with mode
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col].fillna(mode_val, inplace=True)
        print(f"Filled {col} missing values with mode: {mode_val}")

# Verify: no more missing values
print(f"\nTotal missing values remaining: {df.isnull().sum().sum()}")

Filled Postal Code missing values with median: 57103.0
Filled Sales missing values with median: 54.489999999999995
Filled Quantity missing values with median: 3.0
Filled Discount missing values with median: 0.2
Filled Profit missing values with median: 8.6665
Filled Order Date missing values with mode: 9/5/2017
Filled Ship Date missing values with mode: 12/16/2016
Filled Ship Mode missing values with mode: Standard Class
Filled Customer ID missing values with mode: WB-21850
Filled Customer Name missing values with mode: William Brown
Filled Segment missing values with mode: Consumer
Filled Country missing values with mode: United States
Filled City missing values with mode: New York City
Filled State missing values with mode: California
Filled Region missing values with mode: West
Filled Product ID missing values with mode: OFF-PA-10001970
Filled Category missing values with mode: Office Supplies
Filled Sub-Category missing values with mode: Binders
Filled Product Name missing values w

C:\Users\ьшкофдщд\AppData\Local\Temp\ipykernel_15940\1003425343.py:6: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df[col].fillna(median_val, inplace=True)
C:\Users\ьшкофдщд\AppData\Local\Temp\ipykernel_15940\1003425343.py:6: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment u

---
## Task 3: Remove Duplicates (pre-built)

In [12]:
# Check for duplicates
n_duplicates = df.duplicated().sum()
print(f"Duplicate rows found: {n_duplicates}")

# Remove duplicates
if n_duplicates > 0:
    df = df.drop_duplicates()
    print(f"After removal: {df.shape[0]} rows remain")
else:
    print("No duplicates to remove.")

Duplicate rows found: 504
After removal: 10296 rows remain


---
## Task 4: Convert Date Columns (TODO)

The `Order Date` and `Ship Date` columns are stored as strings. Convert them to proper datetime objects.

Hint: Use `pd.to_datetime()`. After conversion, verify with `.dtypes`.

In [13]:
# Check current types of date columns
print("Before conversion:")
for col in df.columns:
    if 'date' in col.lower() or 'Date' in col:
        print(f"  {col}: {df[col].dtype}")
        print(f"  Sample value: {df[col].iloc[0]}")

Before conversion:
  Order Date: str
  Sample value: 11/8/2017
  Ship Date: str
  Sample value: 11/11/2017


In [ ]:
#  Convert date columns to datetime
# Look at the column names printed above and convert them.
# The column names may vary depending on the dataset version.
#
# Example pattern:
# df['Column Name'] = pd.to_datetime(df['Column Name'])


df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])
df.info()

<class 'pandas.DataFrame'>
Index: 10296 entries, 0 to 10798
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Row ID         10296 non-null  str           
 1   Order ID       10296 non-null  str           
 2   Order Date     9994 non-null   datetime64[us]
 3   Ship Date      9994 non-null   datetime64[us]
 4   Ship Mode      9994 non-null   str           
 5   Customer ID    9994 non-null   str           
 6   Customer Name  9994 non-null   str           
 7   Segment        9994 non-null   str           
 8   Country        9994 non-null   str           
 9   City           9994 non-null   str           
 10  State          9994 non-null   str           
 11  Postal Code    9983 non-null   float64       
 12  Region         9994 non-null   str           
 13  Product ID     9994 non-null   str           
 14  Category       9994 non-null   str           
 15  Sub-Category   9994 non-null   str 

In [15]:

# Print dtypes for the date columns to confirm they are datetime64


print(df[['Order Date', 'Ship Date']].dtypes)

Order Date    datetime64[us]
Ship Date     datetime64[us]
dtype: object


---
## Task 5: Derived Features (TODO)

Create customer-level summary features. These are the building blocks for customer segmentation (Activity 4).

You need to create:
- **total_spending**: Total sales per customer
- **order_frequency**: Number of orders per customer
- **avg_order_value**: Average sales amount per order per customer

Hint: Use `df.groupby('Customer ID')` (or whatever the customer ID column is named).

In [17]:
# First, identify the right column names
print("All columns:")
for col in df.columns:
    print(f"  {col}")

All columns:
  Row ID
  Order ID
  Order Date
  Ship Date
  Ship Mode
  Customer ID
  Customer Name
  Segment
  Country
  City
  State
  Postal Code
  Region
  Product ID
  Category
  Sub-Category
  Product Name
  Sales
  Quantity
  Discount
  Profit


In [ ]:
# Create total_spending per customer
# Hint: df.groupby('Customer ID')['Sales'].sum()
#
# Replace column names below with the actual names from your dataset.

# customer_spending = df.groupby('???')['???'].sum()
# customer_spending.name = 'total_spending'

customer_spending = df.groupby('Customer ID')['Sales'].sum()
customer_spending.name = 'total_spending'
print(customer_spending.head())

Customer ID
AA-10315    5563.560
AA-10375    1056.390
AA-10480    1790.512
AA-10645    5086.935
AB-10015     886.156
Name: total_spending, dtype: float64


In [19]:
#  Create order_frequency per customer
# Hint: Count the number of rows (orders) per customer.
# Use .groupby(...).size() or .groupby(...)['some_col'].count()
order_frequency = df.groupby('Customer ID').size()
order_frequency.name = 'order_frequency'
# first five results
print(order_frequency.head())



Customer ID
AA-10315    11
AA-10375    15
AA-10480    12
AA-10645    18
AB-10015     6
Name: order_frequency, dtype: int64


In [20]:
#  Create avg_order_value per customer
# Hint: Use .groupby(...)['Sales'].mean()

avg_order_value = df.groupby('Customer ID')['Sales'].mean()
avg_order_value.name = 'avg_order_value'
print(avg_order_value.head())

Customer ID
AA-10315    505.778182
AA-10375     70.426000
AA-10480    149.209333
AA-10645    282.607500
AB-10015    147.692667
Name: avg_order_value, dtype: float64


In [21]:
#  Combine all three into a single customer-level DataFrame
# Hint: Use pd.concat([series1, series2, series3], axis=1)
# or create them all at once with .groupby(...).agg(...)

# customer_summary = pd.concat([customer_spending, order_freq, avg_order], axis=1)
# customer_summary.columns = ['total_spending', 'order_frequency', 'avg_order_value']

customer_summary = pd.concat([customer_spending, order_frequency, avg_order_value], axis=1)

customer_summary.columns = ['total_spending', 'order_frequency', 'avg_order_value']

# checking
print(customer_summary.head())


             total_spending  order_frequency  avg_order_value
Customer ID                                                  
AA-10315           5563.560               11       505.778182
AA-10375           1056.390               15        70.426000
AA-10480           1790.512               12       149.209333
AA-10645           5086.935               18       282.607500
AB-10015            886.156                6       147.692667


In [22]:
# Display the first 10 rows of your customer summary and .describe()

print("First 10 rows of Customer Summary:")
print(customer_summary.head(10))

print("\n" + "="*30 + "\n")


print("Statistical Summary:")
print(customer_summary.describe())


First 10 rows of Customer Summary:
             total_spending  order_frequency  avg_order_value
Customer ID                                                  
AA-10315           5563.560               11       505.778182
AA-10375           1056.390               15        70.426000
AA-10480           1790.512               12       149.209333
AA-10645           5086.935               18       282.607500
AB-10015            886.156                6       147.692667
AB-10060           7755.620               18       430.867778
AB-10105          14473.571               20       723.678550
AB-10150            966.710               12        80.559167
AB-10165           1113.838               14        79.559857
AB-10255            914.532               14        65.323714


Statistical Summary:
       total_spending  order_frequency  avg_order_value
count      793.000000       793.000000       793.000000
mean      2896.848500        12.602774       227.868165
std       2628.670117         

---
## Task 6: Save Cleaned Data (TODO)

Save the cleaned DataFrame to a new CSV file. Never overwrite the original.

In [23]:
#Save the cleaned main DataFrame
# df.to_csv('superstore_cleaned.csv', index=False)
df.to_csv('superstore_cleaned.csv', index=False)

#Save the customer summary DataFrame
# customer_summary.to_csv('customer_summary.csv')
customer_summary.to_csv('customer_summary.csv')
print("Fayllar muvaffaqiyatli saqlandi!")


Fayllar muvaffaqiyatli saqlandi!


---
## Reflection

Answer these in a text cell or comments:

1. Why did we use median instead of mean for filling numerical missing values?
2. What is the difference between the row-level DataFrame (one row per order) and the customer-level summary? When would you use each?
3. If two rows have identical values in every column, are they always true duplicates? When might they not be?

Reflection Answers
1. Why use median instead of mean for missing values?

Answer: Because the median is resistant to outliers (extreme values). It provides a more accurate representation of the "middle" of the data when noise is present.

2. What is the difference between row-level and customer-level data?

Row-level (df): Represents individual orders. It is used to track specific transactions and products.

Customer-level (summary): Represents individual customers. It is used to analyze customer loyalty and total business impact.

3. Are identical rows always true duplicates?

Answer: Not always. A customer might place two identical orders on the same day. It is important to check the Order ID or timestamp before deleting.